# king − man + woman = ?

---

### La pipeline

```
testo  →  tokenization e dizionario (ID)  →  vettore  →  algebra 
          Modulo 1                           Modulo 2    Modulo 3
```

---
## Setup

La cella qui sotto installa tutto quello che serve e scarica GloVe (~130MB).
La durata per il setup è di circa 2-3 minuti.


In [ ]:
# Clona il repo della lezione e installa le dipendenze
!git clone -q -b scuole https://github.com/tlm-journalclub-org/LLMlego.git
%cd LLMlego
!python setup_colab.py

# Abilita i widget interattivi in Colab (va eseguito DAL kernel del notebook,
# non da setup_colab.py che gira come subprocess)
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # non siamo in Colab

# Importa la libreria didattica della lezione
from llmlego_scuola import *
import numpy as np
print("Tutto pronto!")


---
# Modulo 1: Dal testo ai numeri

Un computer non legge parole, legge **numeri**.
Il primo passo di ogni LLM è trasformare le parole in identificatori interi (token) usando un **dizionario**.

Lo costruiamo noi, partendo da un piccolo corpus a tema tennis.


In [ ]:
corpus_tennis = [
    "Sinner vince la finale",
    "Alcaraz colpisce la palla",
    "Sinner gioca a tennis",
    "il tennis è uno sport bellissimo",
    "Sinner e Alcaraz giocano la finale del torneo",
]


### Esercizio 1.1: Costruisci il dizionario

Ogni parola unica del corpus deve avere un **ID intero**, partendo da 0.

Completa la cella qui sotto. Suggerimento: scorri le frasi, dividile in parole con `.split()`,
e per ogni parola **non ancora vista** aggiungila al dizionario con un nuovo ID.


In [ ]:
dizionario = {}

# TODO: scorri ogni frase del corpus_tennis
# TODO: per ogni parola, se non è già in `dizionario`, aggiungila con un nuovo ID
#       (suggerimento: il prossimo ID disponibile è len(dizionario))

# Il tuo codice qui:


# Visualizziamo il risultato
mostra_dizionario(dizionario)


### Esercizio 1.2: Codifica una frase

Ora che hai il dizionario, scrivi la funzione `codifica` che trasforma una frase in **lista di ID**.


In [ ]:
def codifica(frase):
    # TODO: ritorna la lista degli ID delle parole di `frase`
    pass

# Prova:
print(codifica("Sinner vince la finale"))
# Atteso: una lista di 4 numeri interi


### Cosa succede se la parola **non** è nel dizionario?

Prova a codificare la frase: *"Sinner serve l'ace"*.


In [ ]:
# Esegui questa cella: dovrebbe dare un errore!
codifica("Sinner serve l'ace")

Abbiamo trovato il limite più grande di un dizionario costruito così.
Cosa fa un LLM vero come ChatGPT quando incontra parole sconosciute?

La risposta è semplice e geniale: **spezza le parole in pezzi più piccoli (subword token)**.
Vediamolo coi nostri occhi.


In [ ]:
mostra_token("Sinner serve l'ace")

In [ ]:
# Confronto italiano vs inglese: la stessa frase, due lingue
mostra_token("Il gatto è sul tappeto")
mostra_token("The cat sat on the mat")


In [ ]:
# Parole "inventate" e numeri grandi
mostra_token("supercalifragilistichespiralidoso")
mostra_token("1234567890")

### Sintesi

- Il vocabolario di GPT-4 ha **~100.000 voci** (il nostro corpus ne aveva ~18).
- L'**italiano** usa più token dell'inglese per dire la stessa cosa (i modelli sono allenati su molto più inglese).
- Nessuna parola è davvero "sconosciuta": viene **spezzata in pezzi**.

---

**Domanda chiave per il prossimo modulo:** ok, abbiamo trasformato le parole in numeri.
Ma "re" → 142 e "regina" → 587. Tra `142` e `587` non c'è nessuna relazione di significato.
Per aggiungere le relazioni manca un passaggio: trasformare i numeri in **vettori**.

---
# Modulo 2: Le parole come vettori 2D

Inventiamo noi uno **spazio** in cui mettere le parole. Per ora solo **2 dimensioni**,
quelle che vediamo bene sul piano cartesiano.

- **asse X**: quanto una parola indica una **persona reale** (re, principessa, ecc.)
- **asse Y**: quanto è **femminile**

Sceglierei le coordinate "a occhio" per 6 parole:


In [ ]:
spazio_2d = {
    "re":         ( 0.9, -0.7),
    "regina":     ( 0.9,  0.7),
    "uomo":       (-0.5, -0.7),
    "donna":      (-0.5,  0.7),
    "principe":   ( 0.6, -0.6),
    "principessa":( 0.6,  0.6),
}

plotta_2d(spazio_2d)

### Esercizio 2.1: Aritmetica su carta

Guardate il piano. **Cosa vi aspettate** se fate:

$$\vec{re} - \vec{uomo} + \vec{donna} = \;?$$

Calcolatelo a mano e disegnate il risultato.


In [ ]:
# Trasforma in array numpy per fare aritmetica vettoriale
v_re    = np.array(spazio_2d["re"])
v_uomo  = np.array(spazio_2d["uomo"])
v_donna = np.array(spazio_2d["donna"])

risultato = v_re - v_uomo + v_donna
print(f"Risultato: {risultato}")


In [ ]:
# Visualizziamolo nello spazio (in rosso)
plotta_2d(spazio_2d, evidenzia={"re - uomo + donna": tuple(risultato)})


### Sintesi:

La **relazione "maschile → femminile"** è codificata come una **direzione** nello spazio.
Posso applicarla a qualunque parola.


### Esercizio 2.2: La direzione femminile

Costruisci il **vettore-direzione** `femminile = donna − uomo`, poi applicalo a `principe`.
Dove finisce?


In [ ]:
direzione_femminile = v_donna - v_uomo

v_principe = np.array(spazio_2d["principe"])
principe_femminilizzato = v_principe + direzione_femminile

print(f"Principe + direzione femminile = {principe_femminilizzato}")

plotta_2d(
    spazio_2d,
    evidenzia={"principe + femminile": tuple(principe_femminilizzato)},
)


### La grande domanda

Funziona perché abbiamo scelto noi le coordinate. Ma:

> **Può un computer imparare da solo dove mettere le parole, leggendo testi?**
>
> E se sì, in 2 dimensioni gli bastano?

La risposta è sì, **ma gli servono molte più dimensioni, per esempio ~100**. Vediamole.


---
# Modulo 3: Embedding veri (100 dimensioni)

**GloVe** è un modello che ha letto miliardi di parole inglesi (Wikipedia, news) e ha imparato
a posizionare ogni parola in uno spazio a **100 dimensioni**.

Ogni dimensione non ha un nome leggibile come "regalità" o "femminilità":
sono dimensioni emergenti, scoperte dal modello.

**L'algebra dei vettori funziona uguale.** Quello che abbiamo fatto in 2D lo rifacciamo in 100D.

> 🇮🇹 → 🇬🇧 Da qui in poi lavoriamo in **inglese**: i modelli più studiati sono in inglese
> perché c'è molto più testo disponibile per allenarli.


In [ ]:
# Vediamo com'è fatto il vettore della parola "king"
mostra_vettore("king")

Sopra c'è una "fotografia" del vettore di `king`: 100 dimensioni, alcune positive (rosso) e alcune negative (blu).
Nessuna ha un significato esplicito singolarmente: è la **combinazione** che codifica il significato.


In [ ]:
# Esplora i vicini di una parola: digita una parola inglese qui sotto
widget_vicini("king")


### Esercizio 3.1: Il classico: `king − man + woman = ?`

Costruisci il vettore e cerca la parola più vicina.


In [ ]:
v = vettore("king") - vettore("man") + vettore("woman")
risultato = parola_piu_vicina(v, escludi="king")
print(f"king - man + woman ≈ {risultato}")


### Sintesi:

GloVe non ha mai visto la frase "il re senza la sua mascolinità è una regina".
Eppure il **modello ha imparato la direzione del genere** semplicemente leggendo come queste parole vengono usate.


### Esercizio 3.2: La direzione del genere

Costruisci il vettore-direzione `genere = woman − man` e applicalo ad altre parole.


In [ ]:
direzione_genere = vettore("woman") - vettore("man")

# Applichiamola ad "actor": ci aspettiamo "actress"
nuovo = vettore("actor") + direzione_genere
print(f"actor + (woman - man) ≈ {parola_piu_vicina(nuovo, escludi='actor')}")


### Esercizio 3.3: Prova tu, 3 analogie

Completa le 3 analogie.


In [ ]:
# Analogia 1: capitale -> paese, geografia
# paris : france = ??? : germany
v = vettore("paris") - vettore("france") + vettore("germany")
print("paris - france + germany ≈", parola_piu_vicina(v, escludi="paris"))

# Analogia 2: tempo verbale
# walking : walk = ??? : swim
v = vettore("walking") - vettore("walk") + vettore("swim")
print("walking - walk + swim ≈", parola_piu_vicina(v, escludi="walking"))

# Analogia 3: inventane una tua! Sostituisci le parole qui sotto:
v = vettore("...") - vettore("...") + vettore("...")
# print("la tua analogia ≈", parola_piu_vicina(v, escludi="..."))


---
# Modulo 4: Word Golf

### Regole

1. A coppie. Vi do una **parola di partenza** e una **parola obiettivo (target)**.
2. A ogni mossa scegliete `g.aggiungi("parola")` oppure `g.sottrai("parola")` da una lista fissa.
3. Dopo ogni mossa, la vostra **nuova parola corrente** è quella più vicina al vettore risultante.
4. **Vincete** quando il `target` compare nei **top-5 vicini** del risultato di una mossa.
5. **Vince la coppia con meno mosse.**

Suggerimento dalla matematica: *sottrarre* funziona spesso meglio che *aggiungere*
(perché togliere una direzione ti sposta veramente, mentre aggiungere tende a tirarti
verso la parola che hai aggiunto).


### Prima di iniziare: nome della squadra

Inserite un **nome di squadra**.

In [ ]:
# 1) Scegliete il nome della vostra squadra
NOME_SQUADRA = "I Maestri del Vettore"   # <-- CAMBIATE QUESTO

# 2) Se il prof ha dato un URL della dashboard, incollatelo qui (altrimenti lasciate vuoto)
DASHBOARD_URL = ""   # es. "https://script.google.com/macros/s/.../exec"

# Collega tutto: se DASHBOARD_URL è settato, ogni vittoria viene inviata alla dashboard
import llmlego_scuola.golf as _golf
_golf.DASHBOARD_URL = DASHBOARD_URL or None

if DASHBOARD_URL:
    print(f"✅ Squadra '{NOME_SQUADRA}' collegata alla dashboard!")
else:
    print(f"✅ Squadra '{NOME_SQUADRA}' — classifica solo locale.")


In [ ]:
# Queste sono le ~50 parole disponibili come operatori:
mostra_parole_operatore()

## Round 1 (riscaldamento): `cat → eagle`


In [ ]:
g1 = WordGolf(start="cat", target="eagle", squadra=NOME_SQUADRA)

# Aggiungi/sottrai parole qui sotto. Ad esempio:
# g1.aggiungi("bird")
# g1.sottrai("small")
# g1.stato()


## Round 2: `pizza → emperor`


In [ ]:
g2 = WordGolf(start="pizza", target="emperor", squadra=NOME_SQUADRA)

# Le vostre mosse:


## Round 3 (difficile): `winter → happiness`


In [ ]:
g3 = WordGolf(start="winter", target="happiness", squadra=NOME_SQUADRA)

# Le vostre mosse:


In [ ]:
# Visualizziamo il percorso del round 3 nello spazio (PCA su 2D)
g3.visualizza_percorso()

---
# Cosa abbiamo costruito

Ricapitoliamo: abbiamo visto **tutta la pipeline** di un LLM.

```
testo  →  dizionario (ID)  →  vettore in 100D  →  algebra 
          Modulo 1               Modulo 2         Modulo 3
```

Quello che ChatGPT fa in più è solo: **predire la prossima parola** ripetutamente,
usando una variante più sofisticata di questa algebra (i Transformers, attention, ecc.).
Ma il **cuore è quello che avete appena costruito**.


### ⚠️ Le relazioni nei dati: il bias

Se il modello impara la direzione **genere** leggendo testi reali...
**impara anche i bias che ci sono nei testi reali.**

Proiettiamo alcune professioni sulla direzione `man → woman`:


In [ ]:
mostra_bias_professioni()

**Cosa vedete?** Tutte le professioni "di cura" tendono a `woman`, quelle tecniche/scientifiche a `man`.

Questo NON è un'opinione di GloVe. È uno **specchio** dei testi su cui è stato allenato (Wikipedia, news).
